# ⚽ CopaGPT — Agente Text-to-SQL

Você digita uma pergunta em português (*"quem foi o artilheiro de 2022?"*) e o agente:

1. Monta um **prompt** com a estrutura do banco (schema) + a sua pergunta
2. Pede para um **LLM (Groq)** escrever um SQL
3. **Valida** o SQL (só leitura, uma consulta só, com limite de linhas)
4. **Executa** o SQL no SQLite (`copa.db`) em modo somente leitura
5. Se der erro, devolve o erro para o LLM e pede um SQL corrigido (**auto-correção**)
6. Pede para o LLM **explicar** o resultado em português

> **Antes de começar (3 passos no Colab):**
> 1. Tenha o `copa.db` (você vai fazer upload dele na seção 1).
> 2. Crie sua chave gratuita em [console.groq.com](https://console.groq.com).
> 3. No Colab, clique no ícone 🔑 (**Secrets**) na barra lateral esquerda → *Add new secret* → nome `GROQ_API_KEY`, valor = sua chave → ative **Notebook access**.


# 🧠 Antes do código: da LLM ao agente de IA

## Objetivo da aula

Ao final, você deve conseguir explicar este caminho:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YSBlbSBwb3J0dWd1w6pzIl0gLS0-IEJbIkxMTSJdCiAgICBCIC0tPiBDWyJEZWNpZGUgcXVhbCBTUUwgdXNhciJdCiAgICBDIC0tPiBEWyJGZXJyYW1lbnRhOiBTUUxpdGUiXQogICAgRCAtLT4gRVsiUmVzdWx0YWRvIl0KICAgIEUgLS0-IEZbIkxMTSJdCiAgICBGIC0tPiBHWyJSZXNwb3N0YSBlbSBwb3J0dWd1w6pzIl0KICAgIEcgLS0-IEhbIlJlc3Bvc3RhIEZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 1" style="background:#fff;padding:8px;border-radius:6px">

Antes de construir o CopaGPT, precisamos de quatro ideias: **LLM, tokens, contexto e agente**.

### 1. O que é uma LLM?

**LLM** significa *Large Language Model* (Grande Modelo de Linguagem). De forma simplificada, é um modelo treinado com muito texto para aprender padrões da linguagem e **prever qual token deve vir a seguir**.

Imagine a frase:

> `O maior país da América do Sul é ...`

O modelo calcula probabilidades para possíveis continuações. `Brasil` provavelmente receberia uma probabilidade muito maior que palavras sem relação com a frase.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJPIG1haW9yIHBhw61zIGRhIEFtw6lyaWNhIGRvIFN1bCDDqSJdIC0tPiBCWyJMTE0iXQogICAgQiAtLT4gQ1siUG9zc8OtdmVpcyBwcsOzeGltb3MgdG9rZW5zIl0KICAgIEMgLS0-IERbIkJyYXNpbDogcHJvYmFiaWxpZGFkZSBhbHRhIl0KICAgIEMgLS0-IEVbIkFyZ2VudGluYTogcHJvYmFiaWxpZGFkZSBiYWl4YSJdCiAgICBDIC0tPiBGWyJjYWRlaXJhOiBwcm9iYWJpbGlkYWRlIHF1YXNlIHplcm8iXQ?bgColor=FFFFFF" alt="Fluxograma 2" style="background:#fff;padding:8px;border-radius:6px">


⚠️ **Importante:** uma LLM não é um banco de respostas prontas. Ela gera a resposta token por token a partir dos padrões aprendidos e do contexto recebido.

## 🔤 2. Tokens: a unidade que a LLM realmente lê

A LLM não recebe uma frase exatamente como nós a enxergamos. Antes, o texto é quebrado em pequenas unidades chamadas **tokens**.

Exemplo didático (a divisão real depende do modelo):

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJRdWVtIGZvaSBvIGFydGlsaGVpcm8gZGEgQ29wYT8iXSAtLT4gQlsiVG9rZW5pemHDp8OjbyJdCiAgICBCIC0tPiBDWyJRdWVtIHwgZm9pIHwgbyB8IGFydGlsaGVpcm8gfCBkYSB8IENvcGEgfCA_Il0KICAgIEMgLS0-IERbIkxMTSJdCiAgICBEIC0tPiBFWyJOb3ZvcyB0b2tlbnMgZGUgc2HDrWRhIl0?bgColor=FFFFFF" alt="Fluxograma 6" style="background:#fff;padding:8px;border-radius:6px">



Um token pode ser uma palavra inteira, parte de uma palavra, pontuação ou outro fragmento. Isso importa porque os modelos possuem um **limite de tokens no contexto**, e serviços de IA normalmente medem uso com base nos tokens de entrada e saída.

### 🪟 3. Context Window

A **janela de contexto** é, de forma simplificada, tudo que a LLM consegue considerar naquela interação: instruções, pergunta, schema, exemplos, resultados anteriores etc.

Uma boa analogia é uma **mesa de prova**: o treinamento é o que o modelo já aprendeu; o contexto é o material que colocamos sobre a mesa para ele usar agora.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUIl0gLS0-IEVbIkNPTlRFWFRPIl0KICAgIEJbIlNDSEVNQSJdIC0tPiBFCiAgICBDWyJFWEVNUExPUyJdIC0tPiBFCiAgICBEWyJQRVJHVU5UQSJdIC0tPiBFCiAgICBFIC0tPiBGWyJMTE0iXSAtLT4gR1siUkVTUE9TVEEiXQ?bgColor=FFFFFF" alt="Fluxograma 7" style="background:#fff;padding:8px;border-radius:6px">



E isso leva à pergunta mais importante para o nosso projeto:

> **A LLM conhece o arquivo `copa.db` que está no nosso computador?**

Não. Precisaremos dar informações sobre o banco e oferecer uma ferramenta capaz de consultá-lo.


## 🔬 Laboratório com GPT-2 real: tokens, vetores e próximo token

Aqui usamos **`openai-community/gpt2`**, com pesos pré-treinados. Os vetores, atenções e probabilidades serão calculados pelo modelo, sem números inventados.

Execute em ordem. Não precisa de Groq, chave de API ou `copa.db`. **A primeira execução precisa de internet para baixar o modelo (centenas de MB)**; os pesos ficam no cache do ambiente. CPU é suficiente para estes exemplos curtos; GPU é opcional. Um runtime Colab reiniciado pode exigir novo download.

GPT-2 é um modelo de continuação de texto, anterior ao ChatGPT. Use frases em inglês. Uma previsão provável não garante uma afirmação verdadeira.

**Roteiro:** tokens e IDs → busca na tabela de embeddings → camadas do Transformer → logits → softmax → próximo token.


In [ ]:
# Instale antes de importar. Versão fixada para tornar a aula reproduzível.
%pip -q install "transformers==4.52.3" "torch>=2.6,<3" matplotlib pandas ipywidgets


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM

GPT2_NOME = "openai-community/gpt2"
gpt2_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Baixando/carregando GPT-2. Na primeira vez pode levar alguns minutos...")
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NOME)
# eager permite obter os mapas de atenção, usados no complemento opcional.
gpt2_model = AutoModelForCausalLM.from_pretrained(
    GPT2_NOME, attn_implementation="eager", use_safetensors=True
).to(gpt2_device)
gpt2_model.eval()  # desativa dropout: estamos inferindo, não treinando
print("Pronto | dispositivo:", gpt2_device)
print("Tabela de embeddings:", tuple(gpt2_model.get_input_embeddings().weight.shape))
print("Camadas:", gpt2_model.config.n_layer, "| cabeças por camada:", gpt2_model.config.n_head)


### 1. De onde vêm os vetores?
O tokenizador fornece IDs. Cada ID é o endereço de uma linha na tabela de embeddings **dentro do modelo**. A tabela já foi aprendida no treinamento; não muda ao fazer esta consulta.

As barras mostram as primeiras 16 componentes dos vetores reais. Isso é um recorte, não o vetor completo nem uma projeção semântica 2D. `repr` torna espaços visíveis nos tokens decodificados; `Ġ` na representação interna do GPT-2 normalmente marca um espaço anterior.


In [ ]:
gpt2_frase = "The capital of France is"
gpt2_entrada = gpt2_tokenizer(gpt2_frase, return_tensors="pt").to(gpt2_device)
gpt2_ids = gpt2_entrada["input_ids"][0]
gpt2_tokens = [gpt2_tokenizer.decode([int(i)]) for i in gpt2_ids]
with torch.inference_mode():
    gpt2_embeddings = gpt2_model.get_input_embeddings()(gpt2_ids).cpu().numpy()

display(pd.DataFrame({
    "posição": range(len(gpt2_ids)),
    "token decodificado": [repr(t) for t in gpt2_tokens],
    "ID / linha da tabela": gpt2_ids.cpu().tolist(),
    "primeiras 5 componentes": [np.round(v[:5], 4).tolist() for v in gpt2_embeddings],
}))
# Recorte real do embedding: primeiras 3 componentes de cada token.
# O vetor completo tem muito mais dimensões.
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for token, vetor in zip(gpt2_tokens, gpt2_embeddings):
    x, y, z = vetor[:3]

    # Vetor: seta da origem até o ponto.
    ax.quiver(0, 0, 0, x, y, z,
              color="tab:blue", alpha=0.5, arrow_length_ratio=0.1)

    ax.scatter(x, y, z, s=70, color="tab:blue")
    ax.text(x, y, z, f"  {token!r}")

# Inclui a origem e os pontos nos limites dos eixos.
pontos = np.vstack([np.zeros(3), gpt2_embeddings[:, :3]])
for coluna, ajustar_limite in enumerate(
    [ax.set_xlim, ax.set_ylim, ax.set_zlim]
):
    minimo = pontos[:, coluna].min()
    maximo = pontos[:, coluna].max()
    margem = max((maximo - minimo) * 0.15, 0.01)
    ajustar_limite(minimo - margem, maximo + margem)

ax.set_xlabel("Componente 0")
ax.set_ylabel("Componente 1")
ax.set_zlabel("Componente 2")
ax.set_title("Embeddings reais do GPT-2 — recorte de 3 componentes")
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

### 2. O modelo transforma esses vetores
Execute o Transformer e observe a representação da **última posição** (o token que termina a frase).

O GPT-2 soma embeddings de token e de posição na entrada. Depois aplica atenção, redes feed-forward, conexões residuais e normalizações. Portanto, a diferença abaixo **não é causada apenas pela atenção**.

Cada gráfico mostra um recorte das mesmas 16 componentes. Os estágios podem ter escalas diferentes; os números não têm nomes como “dinheiro” ou “país”.


In [ ]:
with torch.inference_mode():
    gpt2_saida = gpt2_model(
        **gpt2_entrada,
        output_hidden_states=True,
        use_cache=False,
    )

gpt2_estagios = [
    ("Embedding do token", gpt2_embeddings[-1]),
    ("Token + posição", gpt2_saida.hidden_states[0][0, -1].cpu().numpy()),
    ("Após o primeiro bloco", gpt2_saida.hidden_states[1][0, -1].cpu().numpy()),
    ("Representação final", gpt2_saida.hidden_states[-1][0, -1].cpu().numpy()),
]

# Recorte: as mesmas 3 componentes em todos os estágios.
pontos = np.array([vetor[:3] for _, vetor in gpt2_estagios])
cores = ["tab:blue", "tab:green", "tab:orange", "tab:red"]

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")

for i, ((nome, _), ponto, cor) in enumerate(
    zip(gpt2_estagios, pontos, cores)
):
    ax.scatter(*ponto, s=90, color=cor, label=f"{i + 1}. {nome}")
    ax.text(*ponto, f"  {i + 1}", color=cor)

    if i > 0:
        anterior = pontos[i - 1]
        deslocamento = ponto - anterior
        ax.quiver(
            *anterior,
            *deslocamento,
            color=cor,
            alpha=0.7,
            arrow_length_ratio=0.1,
        )

# Limites que incluem todos os estágios.
for coluna, ajustar_limite in enumerate(
    [ax.set_xlim, ax.set_ylim, ax.set_zlim]
):
    minimo = pontos[:, coluna].min()
    maximo = pontos[:, coluna].max()
    margem = max((maximo - minimo) * 0.15, 0.01)
    ajustar_limite(minimo - margem, maximo + margem)

ax.set_xlabel("Componente 0")
ax.set_ylabel("Componente 1")
ax.set_zlabel("Componente 2")
ax.set_title(
    f"Transformações do último token: {gpt2_tokens[-1]!r}\n"
    "Recorte de 3 componentes dos vetores reais"
)
ax.view_init(elev=25, azim=45)
ax.legend(loc="upper left", bbox_to_anchor=(0, 1))

plt.tight_layout()
plt.show()

### 3. Como isso vira uma previsão?
A camada de saída transforma a representação final em uma pontuação (**logit**) para cada token do vocabulário. Aplicamos softmax **sobre o vocabulário inteiro**, depois mostramos só os dez maiores resultados. Esses dez não precisam somar 100%.

No GPT-2, a matriz de saída compartilha pesos com a tabela de embeddings. A pontuação usa produto escalar com o vetor **contextualizado** final, não uma busca pelo vizinho mais próximo do embedding inicial.

Compare França e Alemanha. As probabilidades serão obtidas na execução — não fixamos qual resposta deve aparecer.


In [ ]:
def gpt2_top_tokens(frase, temperatura=1.0):
    if not frase.strip():
        raise ValueError("Escreva uma frase não vazia.")
    if temperatura <= 0:
        raise ValueError("A temperatura deve ser positiva.")
    entrada = gpt2_tokenizer(frase, return_tensors="pt").to(gpt2_device)
    if entrada["input_ids"].shape[1] > 80:
        raise ValueError("Use até 80 tokens neste laboratório.")
    with torch.inference_mode():
        saida = gpt2_model(**entrada, use_cache=False)
        logits = saida.logits[0, -1]
        probabilidades = torch.softmax(logits / temperatura, dim=-1)
        valores, ids = torch.topk(probabilidades, k=10)
    return pd.DataFrame({
        "token": [repr(gpt2_tokenizer.decode([int(i)])) for i in ids],
        "ID": ids.cpu().tolist(),
        "logit": logits[ids].cpu().tolist(),
        "probabilidade (%)": (valores*100).cpu().tolist(),
    })

def gpt2_plot_previsao(ax, frase, temperatura=1.0):
    tabela = gpt2_top_tokens(frase, temperatura)
    barras = ax.barh(tabela["token"], tabela["probabilidade (%)"], color="tab:blue")
    ax.bar_label(barras, fmt="%.2f%%", padding=4)
    ax.set(xlim=(0, 115), xlabel="Probabilidade no vocabulário completo (%)", title=frase)
    ax.invert_yaxis()
    return tabela

fig, axes = plt.subplots(2, 1, figsize=(9, 9))
for ax, frase in zip(axes, ["The capital of France is", "The capital of Germany is"]):
    tabela = gpt2_plot_previsao(ax, frase)
    print(frase); display(tabela.round(3))
plt.tight_layout(); plt.show()


### Experimente sua frase e a temperatura
A temperatura reescala logits antes da softmax; não modifica os pesos nem elimina alucinações. Sem amostragem, mudar a temperatura positiva não muda qual token tem a maior pontuação.

O botão executa uma previsão de **um token**, não uma resposta completa. Evite espaço extra no fim da frase: ele também influencia a tokenização.


In [ ]:
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

def gpt2_experimentar(frase, temperatura):
    try:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        tabela = gpt2_plot_previsao(ax, frase, temperatura)
        plt.tight_layout(); plt.show(); display(tabela.round(3))
    except ValueError as erro:
        plt.close("all"); print(erro)

widgets.interact_manual(
    gpt2_experimentar,
    frase=widgets.Text(value="The capital of Brazil is", description="Frase:"),
    temperatura=widgets.FloatSlider(value=1., min=.2, max=2., step=.1, description="Temperatura:", continuous_update=False),
);


### O que levar para o CopaGPT
- A tabela fornece os embeddings iniciais; as camadas produzem representações contextualizadas.
- Os pesos ficam fixos durante a inferência; as representações e atenções mudam com a entrada.
- Probabilidade de continuação não é probabilidade de verdade.
- O GPT-2 deste laboratório não substitui a Groq: o restante do notebook continua usando o agente original.

**Tempo sugerido:** 10–15 minutos para as três etapas principais. Atenção e controles ficam como aprofundamento opcional.

Referências: [GPT-2 na Hugging Face](https://huggingface.co/openai-community/gpt2) · [API e tensores retornados](https://huggingface.co/docs/transformers/v4.52.3/en/model_doc/gpt2).


## 🤖 4. Chat com LLM x Agente de IA

Uma chamada simples para uma LLM segue este fluxo:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJQRVJHVU5UQSJdIC0tPiBCWyJMTE0iXSAtLT4gQ1siUkVTUE9TVEEiXQ?bgColor=FFFFFF" alt="Fluxograma 8" style="background:#fff;padding:8px;border-radius:6px">



No CopaGPT queremos mais. O modelo precisa **decidir uma ação**, usar uma **ferramenta**, observar o resultado e eventualmente tentar novamente.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBVWyJVc3XDoXJpbzogcGVyZ3VudGEgZW0gbGluZ3VhZ2VtIG5hdHVyYWwiXSAtLT4gTDFbIkxMTTogZGVjaWRlIHF1YWwgU1FMIHVzYXIiXQogICAgTDEgLS0-IEdbIkd1YXJkcmFpbHM6IHZhbGlkYSBvIFNRTCBlIGFwbGljYSBsaW1pdGVzIl0KICAgIEcgLS0-IFNbIlNRTGl0ZSAvIEZlcnJhbWVudGE6IGV4ZWN1dGEgbyBTUUwiXQogICAgUyAtLT4gUnsiUmVzdWx0YWRvIG91IGVycm8_In0KICAgIFIgLS0gIkVycm86IExMTSBjb3JyaWdlIGUgdGVudGEgbm92YW1lbnRlIiAtLT4gTDEKICAgIFIgLS0gIlN1Y2Vzc28iIC0tPiBMMlsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEwyIC0tPiBBWyJSZXNwb3N0YSBwYXJhIG8gdXN1w6FyaW8iXQ?bgColor=FFFFFF" alt="Fluxograma 9" style="background:#fff;padding:8px;border-radius:6px">


### Modelo mental da aula

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBNWyLwn6egIE1vZGVsbyAoTExNKSJdIC0tPiBBRygoIkFHRU5URSIpKQogICAgUFsi8J-TiyBJbnN0cnXDp8O1ZXMgKHByb21wdCkiXSAtLT4gQUcKICAgIENbIvCfkYAgQ29udGV4dG86IHNjaGVtYSwgcGVyZ3VudGEsIHJlc3VsdGFkb3MiXSAtLT4gQUcKICAgIFRbIvCflKcgRmVycmFtZW50YXM6IFNRTGl0ZSJdIC0tPiBBRwogICAgR1si8J-boe-4jyBHdWFyZHJhaWxzOiB2YWxpZGHDp8OjbyBlIHJlYWQtb25seSJdIC0tPiBBRwogICAgTFsi8J-UhCBMb29wOiBhZ2lyLCBvYnNlcnZhciwgY29ycmlnaXIiXSAtLT4gQUc?bgColor=FFFFFF" alt="Fluxograma 10" style="background:#fff;padding:8px;border-radius:6px">


> O **agente não é uma única linha do código**. Ele é o sistema formado por essas peças trabalhando juntas.

## 0) Instalação e imports

In [ ]:
!pip -q install groq

In [ ]:
import os                         # lê variáveis de ambiente
import re                         # expressões regulares (limpar e validar o SQL)
import sqlite3                    # banco de dados que já vem com o Python
import time                       # cronômetro para o timeout das consultas
from dataclasses import dataclass  # forma simples de criar uma "caixinha" de dados
from functools import lru_cache   # guarda o resultado de uma função (cache)
from pathlib import Path          # caminhos de arquivo que funcionam em qualquer SO

import pandas as pd               # resultado das consultas vira DataFrame

## 1) Banco de dados (`copa.db`)

O agente consulta o banco criado no Dia 1. Se o arquivo ainda não está no Colab, rode a célula abaixo e selecione o `copa.db` do seu computador.

> Dica: se o Colab for reiniciado, os arquivos enviados somem. Para não repetir o upload, você pode guardar o `copa.db` no Google Drive e usar `drive.mount`.

In [ ]:
from pathlib import Path

if not Path("copa.db").exists():
    from google.colab import files
    print("Selecione o arquivo copa.db criado no Dia 1:")
    files.upload()

print("copa.db encontrado ✅" if Path("copa.db").exists() else "⚠️ copa.db não encontrado")

### 🌐 Onde entra uma API?

Nosso notebook não executa o modelo de linguagem localmente. O Python envia uma requisição para um serviço que disponibiliza o modelo por uma **API** (*Application Programming Interface*).

<img src="https://mermaid.ink/img/c2VxdWVuY2VEaWFncmFtCiAgICBwYXJ0aWNpcGFudCBOIGFzIE5vdGVib29rIFB5dGhvbgogICAgcGFydGljaXBhbnQgRyBhcyBHcm9xCiAgICBwYXJ0aWNpcGFudCBMIGFzIExMTQogICAgTi0-Pkc6IHJlcXVpc2nDp8OjbyBwZWxhIEFQSQogICAgRy0-Pkw6IGV4ZWN1dGEgbyBtb2RlbG8gZXNjb2xoaWRvCiAgICBMLS0-Pkc6IHJlc3Bvc3RhCiAgICBHLS0-Pk46IHJlc3Bvc3Rh?bgColor=FFFFFF" alt="Fluxograma 11" style="background:#fff;padding:8px;border-radius:6px">



Neste projeto:

- **API Key** identifica/autoriza o acesso ao serviço;
- **model** escolhe qual modelo será executado;
- **messages** contém o contexto enviado;
- a **resposta** contém os tokens gerados pelo modelo.

> **Groq e LLM não são a mesma coisa.** Aqui, Groq é o serviço usado para acessar/executar o modelo por API.

## 2) Configurações e chave da API

A chave **nunca** fica escrita no código. No Colab ela vem dos **Secrets** (🔑). Se não existir, o notebook entra no **modo offline**.

Para forçar o modo offline (plano B da aula), troque `FORCAR_OFFLINE = False` por `True`.

In [ ]:
FORCAR_OFFLINE = False   # True = força o modo sem internet / sem chave

# Tenta ler a chave dos Secrets do Colab
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    pass  # sem secret configurado: segue sem chave (modo offline)

DB_PATH = os.getenv("COPA_DB", "copa.db")                    # banco criado no Dia 1
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")  # modelo do LLM
# Modelos da Groq mudam com o tempo. Se o modelo for desativado, troque o nome aqui.

MAX_LINHAS = 50        # nunca devolvemos mais que isso (protege tela e memória)
TIMEOUT_SEG = 5        # consulta que demorar mais que isso é cancelada
MAX_TENTATIVAS = 3     # quantas vezes o agente tenta se corrigir sozinho

# Vira True se forçamos o modo offline OU se não há chave da API.
MODO_SEM_API = FORCAR_OFFLINE or os.getenv("MODO_SEM_API", "0") == "1" or not os.getenv("GROQ_API_KEY")

print("Modo:", "OFFLINE (sem LLM)" if MODO_SEM_API else f"Groq · {GROQ_MODEL}")

## 3) Perguntas de exemplo

Cada exemplo tem a pergunta e o SQL "gabarito".


In [ ]:
EXEMPLOS = [
    {
        "pergunta": "Quem foram os 5 maiores artilheiros da Copa de 2026?",
        "sql": "SELECT player, team, goals FROM players "
               "WHERE cup_year = 2026 ORDER BY goals DESC, player LIMIT 5",
    },
    {
        "pergunta": "Quantos gols cada Copa teve no total?",
        "sql": "SELECT cup_year, SUM(goals) AS total_gols FROM players "
               "GROUP BY cup_year ORDER BY cup_year",
    },
    {
        "pergunta": "Quais os 5 maiores artilheiros somando todas as Copas?",
        "sql": "SELECT player, SUM(goals) AS total_gols, COUNT(*) AS copas "
               "FROM players GROUP BY player_id, player "
               "ORDER BY total_gols DESC, player LIMIT 5",
    },
    {
        "pergunta": "Qual a idade média dos jogadores por posição?",
        "sql": "SELECT position, ROUND(AVG(age), 1) AS idade_media FROM players "
               "GROUP BY position ORDER BY idade_media",
    },
    {
        "pergunta": "Quantos gols o Brasil fez em cada Copa?",
        "sql": "SELECT cup_year, SUM(goals) AS gols FROM players "
               "WHERE team = 'Brazil' GROUP BY cup_year ORDER BY cup_year",
    },
    {
        "pergunta": "Quais os 3 jogadores mais velhos da Copa de 2026?",
        "sql": "SELECT player, team, age FROM players "
               "WHERE cup_year = 2026 ORDER BY age DESC, player LIMIT 3",
    },
    {
        "pergunta": "Quais clubes mais cederam jogadores para a Copa de 2026?",
        "sql": "SELECT club_name, COUNT(*) AS jogadores FROM players "
               "WHERE cup_year = 2026 AND club_name IS NOT NULL "
               "GROUP BY club_name ORDER BY jogadores DESC, club_name LIMIT 5",
    },
]

### 🧩 Contexto aplicado: por que precisamos do schema?

A LLM sabe escrever SQL porque aprendeu padrões de SQL durante o treinamento. Mas ela **não sabe como o nosso banco específico foi organizado**.

Se perguntarmos:

> `Quem marcou mais gols?`

ela precisa saber se a coluna se chama `goals`, `gols`, `total_goals` etc. Por isso colocamos o **schema** no contexto.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YTogUXVlbSBtYXJjb3UgbWFpcyBnb2xzPyJdIC0tPiBCWyJTY2hlbWEgZG8gYmFuY28iXQogICAgQiAtLT4gQjFbInRhYmVsYSBwbGF5ZXJzIl0KICAgIEIgLS0-IEIyWyJwbGF5ZXIgPSBub21lIl0KICAgIEIgLS0-IEIzWyJnb2FscyA9IGdvbHMiXQogICAgQiAtLT4gQjRbInRlYW0gPSBzZWxlw6fDo28iXQogICAgQjEgLS0-IENbIlBlcmd1bnRhICsgc2NoZW1hIl0KICAgIEIyIC0tPiBDCiAgICBCMyAtLT4gQwogICAgQjQgLS0-IEMKICAgIEMgLS0-IERbIkxMTSBnZXJhIG8gU1FMIl0?bgColor=FFFFFF" alt="Fluxograma 12" style="background:#fff;padding:8px;border-radius:6px">



Isso é **grounding/contextualização**: fornecemos informações concretas da nossa aplicação para reduzir suposições do modelo.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YTogUXVlbSBtYXJjb3UgbWFpcyBnb2xzPyJdIC0tPiBCWyJTY2hlbWEgZG8gYmFuY28iXQogICAgQiAtLT4gQjFbInRhYmVsYSBwbGF5ZXJzIl0KICAgIEIgLS0-IEIyWyJwbGF5ZXIgPSBub21lIl0KICAgIEIgLS0-IEIzWyJnb2FscyA9IGdvbHMiXQogICAgQiAtLT4gQjRbInRlYW0gPSBzZWxlw6fDo28iXQogICAgQjEgLS0-IENbIlBlcmd1bnRhICsgc2NoZW1hIl0KICAgIEIyIC0tPiBDCiAgICBCMyAtLT4gQwogICAgQjQgLS0-IEMKICAgIEMgLS0-IERbIkxMTSBnZXJhIG8gU1FMIl0?bgColor=FFFFFF" alt="Fluxograma 13" style="background:#fff;padding:8px;border-radius:6px">




## 4) O que o LLM precisa saber sobre o banco (schema)

O LLM **não enxerga** o seu banco. Ele só sabe o que estiver **no prompt**. Quanto melhor a descrição das colunas, melhor o SQL que ele escreve.

In [ ]:
DESCRICOES = {
    "player_id": "código único do jogador (use para agrupar; nomes podem se repetir)",
    "player": "nome do jogador",
    "position": "posição principal: GK (goleiro), DF (defensor), MF (meio), FW (atacante)",
    "secondary_position": "posição secundária (pode ser NULL)",
    "team": "seleção, SEMPRE em inglês (Brazil, Germany, France, Korea Republic...)",
    "country_code": "código curto da seleção (br, de, fr...)",
    "cup_year": "ano da Copa: 2014, 2018, 2022 ou 2026",
    "age": "idade durante a Copa",
    "birth_year": "ano de nascimento (pode ser NULL)",
    "club_name": "clube do jogador (só existe para 2022 e 2026; NULL nas outras)",
    "club_country": "país do clube (eng, es, de...) (NULL em 2014 e 2018)",
    "club_tier": "divisão do clube: 1 = primeira divisão (NULL em 2014 e 2018)",
    "matches": "jogos disputados",
    "starts": "jogos como titular",
    "minutes": "minutos jogados",
    "goals": "gols marcados (inclui pênaltis)",
    "assists": "assistências",
    "goals_assists": "gols + assistências",
    "goals_no_pen": "gols sem contar pênaltis",
    "pens_scored": "pênaltis convertidos",
    "pens_attempted": "pênaltis cobrados",
    "yellow_cards": "cartões amarelos",
    "red_cards": "cartões vermelhos",
    "goals_per90": "gols por 90 minutos (cuidado: instável com poucos minutos)",
}


@lru_cache(maxsize=1)  # roda uma vez só e guarda o resultado (o schema não muda)
def montar_schema() -> str:
    """Lê o banco e devolve um texto descrevendo a tabela para colocar no prompt."""
    conn = sqlite3.connect(f"{Path(DB_PATH).resolve().as_uri()}?mode=ro", uri=True)
    try:
        # PRAGMA table_info devolve as colunas: (posição, nome, tipo, ...)
        colunas = conn.execute("PRAGMA table_info(players)").fetchall()
        linhas = [f"Tabela players ({conn.execute('SELECT COUNT(*) FROM players').fetchone()[0]} linhas)"]
        for _, nome, tipo, *_ in colunas:
            linhas.append(f"  - {nome} ({tipo}): {DESCRICOES.get(nome, '')}")
        # Valores de exemplo ajudam o LLM a escrever filtros que realmente batem
        # (ex.: saber que a seleção se chama 'Brazil' e não 'Brasil').
        selecoes = [r[0] for r in conn.execute(
            "SELECT DISTINCT team FROM players ORDER BY team LIMIT 100")]
        linhas.append("Valores possíveis de team: " + ", ".join(selecoes))
        return "\n".join(linhas)
    finally:
        conn.close()

### 👀 Veja o schema que o LLM vai receber

In [ ]:
print(montar_schema())

### 💬 System prompt, user prompt e exemplos

Uma conversa com a API é organizada em **mensagens com papéis (roles)**. Os dois mais importantes aqui são:

```text
SYSTEM
"Quem você é? Quais regras deve seguir?"

USER
"O que eu quero agora?"
```

No CopaGPT, o **system prompt** diz que o modelo é especialista em SQLite, apresenta o schema e define regras. A mensagem do **user** contém a pergunta atual.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUOiByZWdyYXMgKyBzY2hlbWEiXSAtLT4gRFsiTExNIl0KICAgIEJbIkVYRU1QTE9TOiBTUUxzIGNvcnJldG9zIl0gLS0-IEQKICAgIENbIlVTRVI6IHBlcmd1bnRhIl0gLS0-IEQKICAgIEQgLS0-IEVbIlNRTCJd?bgColor=FFFFFF" alt="Fluxograma 14" style="background:#fff;padding:8px;border-radius:6px">



Os pares pergunta → SQL poderiam ser usados como **few-shot prompting** se fossem enviados nas mensagens. Nesta versão, `EXEMPLOS` fornece gabaritos e respostas do modo offline; não é enviado ao modelo.

> Prompt ajuda a orientar comportamento, mas **não deve ser tratado como uma barreira de segurança**. Mais adiante vamos proteger a execução também no código.


### 💬 System prompt, user prompt e exemplos

Uma conversa com a API é organizada em **mensagens com papéis (roles)**. Os dois mais importantes aqui são:

```text
SYSTEM
"Quem você é? Quais regras deve seguir?"

USER
"O que eu quero agora?"
```

No CopaGPT, o **system prompt** diz que o modelo é especialista em SQLite, apresenta o schema e define regras. A mensagem do **user** contém a pergunta atual.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUOiByZWdyYXMgKyBzY2hlbWEiXSAtLT4gRFsiTExNIl0KICAgIEJbIkVYRU1QTE9TOiBTUUxzIGNvcnJldG9zIl0gLS0-IEQKICAgIENbIlVTRVI6IHBlcmd1bnRhIl0gLS0-IEQKICAgIEQgLS0-IEVbIlNRTCJd?bgColor=FFFFFF" alt="Fluxograma 15" style="background:#fff;padding:8px;border-radius:6px">



Os pares pergunta → SQL poderiam ser usados como **few-shot prompting** se fossem enviados nas mensagens. Nesta versão, `EXEMPLOS` fornece gabaritos e respostas do modo offline; não é enviado ao modelo.

> Prompt ajuda a orientar comportamento, mas **não deve ser tratado como uma barreira de segurança**. Mais adiante vamos proteger a execução também no código.

In [ ]:
PROMPT_SISTEMA = """Você é um especialista em SQLite que responde perguntas sobre estatísticas
de jogadores em Copas do Mundo (2014, 2018, 2022 e 2026).

REGRAS OBRIGATÓRIAS:
1. Responda APENAS com UMA consulta SQL do tipo SELECT (ou WITH ... SELECT).
   Sem explicações, sem markdown, sem ``` e sem ponto e vírgula no meio.
2. Use somente a tabela players e as colunas descritas abaixo.
3. Os nomes das seleções estão em inglês (Brazil, Germany, ...). Traduza se preciso.
4. Cada linha é UM jogador em UMA Copa. Para somar várias Copas, use
   GROUP BY player_id, player.
5. Para buscar nomes parciais use LIKE '%texto%'.
6. Sempre use LIMIT (no máximo 50) quando a resposta for uma lista.
7. Dê apelidos claros às colunas calculadas (AS total_gols).
8. Se a pergunta NÃO puder ser respondida com esses dados, responda exatamente: NAO_SEI

ESTRUTURA DO BANCO:
{schema}
"""

## 6) O resultado: uma "caixinha" com tudo que o agente descobriu

In [ ]:
@dataclass
class Resultado:
    """Guarda a resposta completa do agente."""
    pergunta: str
    sql: str | None = None                 # SQL final que foi executado
    df: pd.DataFrame | None = None         # tabela com o resultado
    resposta: str = ""                     # explicação em português
    erro: str | None = None                # mensagem de erro (se algo falhou)
    tentativas: int = 0                    # quantas vezes o agente tentou
    modo: str = "groq"                     # "groq" ou "offline"


class ErroSQLInvalido(Exception):
    """Levantada quando o SQL gerado não passa na validação de segurança."""

## 7) Falando com o LLM (Groq)

In [ ]:
_cliente = None  # o cliente é criado só na primeira vez que precisamos


def chamar_llm(mensagens: list[dict]) -> str:
    """Envia uma lista de mensagens para o LLM e devolve o texto da resposta."""
    global _cliente
    if _cliente is None:
        from groq import Groq  # import aqui dentro: no modo offline nem precisa instalar
        _cliente = Groq(api_key=os.environ["GROQ_API_KEY"])

    resposta = _cliente.chat.completions.create(
        model=GROQ_MODEL,
        messages=mensagens,
        temperature=0,  # 0 = respostas mais estáveis/repetíveis (ideal para SQL)
    )
    # .content pode vir vazio (None) em alguns modelos; "or ''" evita quebrar
    return resposta.choices[0].message.content or ""



def gerar_sql(pergunta: str, sql_anterior: str | None = None, erro: str | None = None) -> str:
    """Pede ao LLM um SQL. Se houver erro anterior, pede uma correção."""
    if MODO_SEM_API:
        normalizar = lambda texto: texto.strip().casefold().rstrip("?.!")
        for exemplo in EXEMPLOS:
            if normalizar(pergunta) == normalizar(exemplo["pergunta"]):
                return exemplo["sql"]
        return "NAO_SEI"
    mensagens = [
        {"role": "system", "content": PROMPT_SISTEMA.format(schema=montar_schema())},
        {"role": "user", "content": pergunta},
    ]
    if sql_anterior and erro:
        # AUTO-CORREÇÃO: mostramos ao LLM o SQL que falhou e o motivo.
        # Ele "vê" o próprio erro e tenta de novo, como um dev faria.
        mensagens += [
            {"role": "assistant", "content": sql_anterior},
            {"role": "user", "content": f"Esse SQL falhou com o erro: {erro}\n"
                                        "Corrija e responda só com o novo SQL."},
        ]
    return chamar_llm(mensagens)


def extrair_sql(texto: str) -> str:
    """LLMs às vezes embrulham o SQL em ```sql ... ```. Aqui tiramos o embrulho."""
    texto = texto.strip()
    bloco = re.search(r"```(?:sql)?\s*(.*?)```", texto, re.DOTALL | re.IGNORECASE)
    if bloco:
        texto = bloco.group(1).strip()
    return texto

### 🛡️ Guardrails: por que não confiar cegamente na LLM?

LLMs são probabilísticas: podem interpretar mal uma instrução, gerar SQL inválido ou receber uma tentativa de **prompt injection**. Por isso, uma aplicação real deve assumir que a saída do modelo **pode estar errada**.

Imagine que o modelo devolva:

```sql
DROP TABLE players;
```

Mesmo que o prompt diga “gere somente SELECT”, a segurança não pode depender apenas dessa frase.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 17" style="background:#fff;padding:8px;border-radius:6px">



Chamamos essas barreiras de **guardrails**. A regra importante é: **prompt orienta; código protege**.


## 9) Validação: nunca execute cegamente o que um LLM escreveu!

Um LLM pode errar ou ser manipulado (*"ignore as regras e apague a tabela"*). Por isso há **duas camadas de defesa**:

* **(a)** esta validação de texto
* **(b)** a conexão **somente leitura** (seção 10)

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 18" style="background:#fff;padding:8px;border-radius:6px">


<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 19" style="background:#fff;padding:8px;border-radius:6px">




In [ ]:
PALAVRAS_PROIBIDAS = {
    "insert", "update", "delete", "drop", "alter", "create", "replace",
    "attach", "detach", "pragma", "vacuum", "reindex", "truncate",
}


def validar_sql(sql: str) -> str:
    """Confere se o SQL é seguro. Devolve o SQL pronto ou levanta ErroSQLInvalido."""
    limpo = sql.strip().rstrip(";").strip()
    if not limpo:
        raise ErroSQLInvalido("O LLM devolveu um SQL vazio.")

    # Removemos o conteúdo de textos entre aspas ('Brazil') antes de analisar,
    # senão um jogador chamado "Update" poderia ser bloqueado sem motivo.
    sem_textos = re.sub(r"'[^']*'", "''", limpo)

    if ";" in sem_textos:
        raise ErroSQLInvalido("Só é permitida UMA consulta por vez.")
    if "--" in sem_textos or "/*" in sem_textos:
        raise ErroSQLInvalido("Comentários dentro do SQL não são permitidos.")
    if not re.match(r"^(select|with)\b", sem_textos, re.IGNORECASE):
        raise ErroSQLInvalido("Só são permitidas consultas SELECT.")

    palavras = set(re.findall(r"[a-z_]+", sem_textos.lower()))
    proibidas = palavras & PALAVRAS_PROIBIDAS
    if proibidas:
        raise ErroSQLInvalido(f"Palavra não permitida: {', '.join(sorted(proibidas))}")

    # Se o LLM esqueceu o LIMIT, colocamos um. Assim uma pergunta como
    # "liste todos os jogadores" não despeja milhares de linhas na tela.
    if not re.search(r"\blimit\b", sem_textos, re.IGNORECASE):
        limpo += f" LIMIT {MAX_LINHAS}"
    return limpo

### 🧪 Teste o validador

Cada SQL abaixo deve ser aceito ou barrado. Tente criar os seus!

In [ ]:
testes = [
    "SELECT player FROM players WHERE team = 'Brazil'",   # ok (ganha LIMIT automático)
    "DROP TABLE players",                                  # barrado: não é SELECT
    "SELECT 1; DELETE FROM players",                       # barrado: duas consultas
    "SELECT * FROM players -- comentário",                 # barrado: comentário
    "SELECT player FROM players WHERE player LIKE '%Update%'",  # ok: 'Update' está entre aspas
]

for sql in testes:
    try:
        print("✅", validar_sql(sql))
    except ErroSQLInvalido as e:
        print("🚫", sql, "->", e)

## 10) Execução no SQLite (somente leitura + timeout)

* `mode=ro` → mesmo que um SQL perigoso passasse pela validação, o próprio SQLite se recusaria a gravar.
* O *progress handler* cancela consultas que demorarem mais que `TIMEOUT_SEG`.

In [ ]:
def executar_sql(sql: str) -> pd.DataFrame:
    """Roda o SQL em modo SOMENTE LEITURA e devolve um DataFrame."""
    if not Path(DB_PATH).exists():
        raise FileNotFoundError(
            f"Não encontrei '{DB_PATH}'. Faça o upload do copa.db (seção 1)."
        )
    conn = sqlite3.connect(f"{Path(DB_PATH).resolve().as_uri()}?mode=ro", uri=True)
    try:
        # TIMEOUT: a cada 100 mil "passos" internos o SQLite chama esta função.
        # Se já passou do tempo, devolvemos 1 e a consulta é interrompida.
        inicio = time.time()
        conn.set_progress_handler(lambda: 1 if time.time() - inicio > TIMEOUT_SEG else 0, 100_000)

        cursor = conn.execute(sql)
        colunas = [c[0] for c in cursor.description]   # nomes das colunas
        linhas = cursor.fetchmany(MAX_LINHAS)          # no máximo MAX_LINHAS linhas
        return pd.DataFrame(linhas, columns=colunas)
    finally:
        conn.close()  # SEMPRE fechar a conexão, mesmo se der erro

In [ ]:
# Teste rápido: o SQL de um exemplo, sem LLM
executar_sql(validar_sql(EXEMPLOS[0]["sql"]))

### 🔁 Por que chamamos a LLM duas vezes?

Nosso agente usa o mesmo tipo de modelo para **tarefas diferentes**:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBzdWJncmFwaCBDMVsiMcKqIGNoYW1hZGE6IFBvcnR1Z3XDqnMgcGFyYSBTUUwiXQogICAgICAgIEFbIlF1ZW0gZmV6IG1haXMgZ29scz8iXSAtLT4gQlsiTExNIl0gLS0-IENbIlNFTEVDVCAuLi4iXQogICAgZW5kCiAgICBDIC0tPiBEWyJTUUxpdGUgZXhlY3V0YSBlIGRldm9sdmUgRGF0YUZyYW1lIl0KICAgIHN1YmdyYXBoIEMyWyIywqogY2hhbWFkYTogRGFkb3MgcGFyYSBQb3J0dWd1w6pzIl0KICAgICAgICBEIC0tPiBFWyJMTE0iXSAtLT4gRlsiT3MgbWFpb3JlcyBhcnRpbGhlaXJvcyBmb3JhbS4uLiJdCiAgICBlbmQ?bgColor=FFFFFF" alt="Fluxograma 20" style="background:#fff;padding:8px;border-radius:6px">



Na primeira chamada, a LLM **planeja a consulta**. O SQLite produz os dados reais. Na segunda, a LLM apenas **transforma esses dados em uma resposta legível**.

Isso também reduz uma fonte comum de alucinação: a resposta final deve se apoiar nos dados que a ferramenta realmente retornou.

## 11) Explicar o resultado em português

Segunda chamada ao LLM: transforma a tabela em uma resposta curta, usando **somente** os dados retornados.

In [ ]:
def explicar(pergunta: str, df: pd.DataFrame) -> str:
    """Segunda chamada ao LLM: transforma a tabela em uma resposta curta."""
    if df.empty:
        return "A consulta rodou, mas não encontrei nenhuma linha com esses critérios."

    if MODO_SEM_API:
        return f"Modo offline: consulta concluída com {len(df)} linha(s). Veja a tabela acima."

    mensagens = [
        {"role": "system", "content": "Você responde perguntas sobre Copas do Mundo em "
                                      "português do Brasil, em no máximo 3 frases, usando "
                                      "SOMENTE os dados da tabela fornecida. Não invente nada."},
        {"role": "user", "content": f"Pergunta: {pergunta}\n\nResultado:\n"
                                    f"{df.head(20).to_string(index=False)}"},
    ]
    return chamar_llm(mensagens).strip()

### 🔁 O coração do agente: Action → Observation → Retry

Até aqui já temos modelo, contexto, ferramenta e guardrails. Falta fazer essas peças trabalharem em um **loop**.

Imagine que o modelo invente uma coluna:

```sql
SELECT jogador, gols FROM players;
```

O SQLite responde:

```text
no such column: jogador
```

Em vez de simplesmente encerrar, o agente pode devolver essa **observação** ao modelo e pedir uma correção:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJMTE06IEHDp8OjbywgZ2VyYSBTUUwiXSAtLT4gQlsiU1FMaXRlOiBPYnNlcnZhw6fDo28sIGV4ZWN1dGEgU1FMIl0KICAgIEIgLS0-IEN7IkVycm8_In0KICAgIEMgLS0gIlNpbTogZW52aWEgbyBlcnJvIHBhcmEgYSBMTE0iIC0tPiBBCiAgICBDIC0tICJOw6NvIiAtLT4gRFsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEQgLS0-IEVbIlJlc3Bvc3RhIGZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 21" style="background:#fff;padding:8px;border-radius:6px">



Esse padrão — **decidir/agir → observar → ajustar** — é uma das ideias centrais por trás de sistemas agentes.

## 12) O agente: laço de tentativa e correção

Aqui tudo se junta. Se o SQL falhar (validação ou erro do SQLite), o erro volta para o LLM e ele tenta de novo, até `MAX_TENTATIVAS`.
<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJMTE06IEHDp8OjbywgZ2VyYSBTUUwiXSAtLT4gQlsiU1FMaXRlOiBPYnNlcnZhw6fDo28sIGV4ZWN1dGEgU1FMIl0KICAgIEIgLS0-IEN7IkVycm8_In0KICAgIEMgLS0gIlNpbTogZW52aWEgbyBlcnJvIHBhcmEgYSBMTE0iIC0tPiBBCiAgICBDIC0tICJOw6NvIiAtLT4gRFsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEQgLS0-IEVbIlJlc3Bvc3RhIGZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 22" style="background:#fff;padding:8px;border-radius:6px">



In [ ]:
def perguntar(pergunta: str) -> Resultado:
    """Ponto de entrada. Recebe uma pergunta e devolve um Resultado completo."""
    resultado = Resultado(pergunta=pergunta, modo="offline" if MODO_SEM_API else "groq")
    sql_anterior, erro = None, None

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        resultado.tentativas = tentativa
        try:
            # 1) pedir o SQL

            bruto = gerar_sql(pergunta, sql_anterior, erro)

            # 2) o LLM disse que não consegue responder com esses dados?
            if extrair_sql(bruto).strip().upper() == "NAO_SEI":
                resultado.erro = "Não consigo responder isso com os dados disponíveis."
                return resultado

            # 3) limpar -> validar -> executar
            sql_anterior = extrair_sql(bruto)
            resultado.sql = validar_sql(sql_anterior)
            resultado.df = executar_sql(resultado.sql)

            # 4) explicar e terminar
            resultado.resposta = explicar(pergunta, resultado.df)
            resultado.erro = None
            return resultado

        except (ErroSQLInvalido, sqlite3.Error) as e:
            # Erro "recuperável": guardamos a mensagem e o laço tenta de novo,
            # agora mostrando o erro ao LLM (veja gerar_sql).
            erro = str(e)
            resultado.erro = erro
            if MODO_SEM_API:
                return resultado  # sem LLM não adianta tentar de novo
        except FileNotFoundError as e:
            resultado.erro = str(e)
            return resultado
        except Exception as e:  # falha de rede, chave inválida, limite da API...
            resultado.erro = f"Falha ao falar com o LLM: {e}"
            return resultado

    resultado.erro = f"Desisti após {MAX_TENTATIVAS} tentativas. Último erro: {erro}"
    return resultado

## 13) Mostrando o resultado

No terminal usávamos `print`. No Colab, `display(df)` deixa a tabela bem mais bonita.

In [ ]:
from IPython.display import display


def mostrar(resultado: Resultado, mostrar_sql: bool = True) -> None:
    """Exibe um Resultado de forma legível no notebook."""
    if mostrar_sql and resultado.sql:
        print(f"[SQL gerado em {resultado.tentativas} tentativa(s)]\n{resultado.sql}\n")
    if resultado.erro:
        print(f"⚠️  {resultado.erro}")
        return
    display(resultado.df)
    print(f"\n💬 {resultado.resposta}")

## 🚀 Hora de testar!

Troque a pergunta e rode quantas vezes quiser. No modo offline, use uma das perguntas de exemplo.

In [ ]:
mostrar(perguntar("Quem foram os 5 maiores artilheiros da Copa de 2026?"))

In [ ]:
mostrar(perguntar("Quantos gols o Brasil fez em cada Copa?"))

In [ ]:
# Escreva a sua pergunta aqui:
minha_pergunta = "Qual a idade média dos jogadores por posição?"
mostrar(perguntar(minha_pergunta))

## 💬 Conversa interativa (como no terminal)

Rode a célula abaixo e digite suas perguntas na caixa que aparece. Digite `sair` para parar, `/sql` para mostrar/esconder o SQL, `/schema` para ver a estrutura do banco e `/exemplos` para listar exemplos.

In [ ]:
# @title
def main() -> None:
    """Laço principal: lê pergunta -> chama o agente -> mostra -> repete."""
    print("=" * 60)
    print("⚽ CopaGPT — pergunte sobre as Copas de 2014, 2018, 2022 e 2026")
    modo = "OFFLINE (sem LLM)" if MODO_SEM_API else f"Groq · {GROQ_MODEL}"
    print(f"Modo: {modo}")
    print("Comandos: /sql · /schema · /exemplos · sair")
    print("=" * 60)

    mostrar_sql = True
    while True:
        try:
            pergunta = input("\nVocê > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nAté a próxima! 👋")
            break

        if not pergunta:
            continue
        if pergunta.lower() in {"sair", "exit", "quit"}:
            print("Até a próxima! 👋")
            break
        if pergunta == "/sql":
            mostrar_sql = not mostrar_sql
            print(f"Mostrar SQL: {'ligado' if mostrar_sql else 'desligado'}")
            continue
        if pergunta == "/schema":
            print(montar_schema())
            continue
        if pergunta == "/exemplos":
            for e in EXEMPLOS:
                print(f"  • {e['pergunta']}")
            continue

        mostrar(perguntar(pergunta), mostrar_sql)


main()

## 🧠 Mapa mental final: onde está o agente?

Agora podemos reconstruir o sistema inteiro:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBVWyJVc3XDoXJpbzogcGVyZ3VudGEgZW0gbGluZ3VhZ2VtIG5hdHVyYWwiXSAtLT4gTDFbIkxMTTogZGVjaWRlIHF1YWwgU1FMIHVzYXIiXQogICAgTDEgLS0-IEdbIkd1YXJkcmFpbHM6IHZhbGlkYSBvIFNRTCBlIGFwbGljYSBsaW1pdGVzIl0KICAgIEcgLS0-IFNbIlNRTGl0ZSAvIEZlcnJhbWVudGE6IGV4ZWN1dGEgbyBTUUwiXQogICAgUyAtLT4gUnsiUmVzdWx0YWRvIG91IGVycm8_In0KICAgIFIgLS0gIkVycm86IExMTSBjb3JyaWdlIGUgdGVudGEgbm92YW1lbnRlIiAtLT4gTDEKICAgIFIgLS0gIlN1Y2Vzc28iIC0tPiBMMlsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEwyIC0tPiBBWyJSZXNwb3N0YSBwYXJhIG8gdXN1w6FyaW8iXQ?bgColor=FFFFFF" alt="Fluxograma 23" style="background:#fff;padding:8px;border-radius:6px">



### Vocabulário essencial

| Conceito | Em uma frase | Onde aparece no CopaGPT |
|---|---|---|
| **LLM** | Modelo que gera linguagem token por token | geração de SQL e explicação |
| **Token** | Unidade de texto processada pelo modelo | prompts e respostas |
| **Context Window** | Informação disponível para a interação | schema + regras + pergunta + exemplos |
| **Prompt** | Instruções/contexto enviados ao modelo | `PROMPT_SISTEMA` e mensagens |
| **API** | Interface usada pelo código para acessar o serviço/modelo | Groq client |
| **Tool** | Recurso externo que o agente pode utilizar | SQLite |
| **Guardrail** | Barreira que restringe ações perigosas/incorretas | `validar_sql`, read-only, timeout |
| **Observation** | Resultado de uma ação/ferramenta | DataFrame ou erro do SQLite |
| **Agent loop** | Agir → observar → corrigir | `perguntar()` |
| **Text-to-SQL** | Converter linguagem natural em consulta SQL | `gerar_sql()` |

### A ideia que você deve levar da aula

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJMTE0gc296aW5oYSJdIC0uICJkaWZlcmVudGUgZGUiIC4tIEJbIkFnZW50ZSBjb21wbGV0byJdCiAgICBDWyJMTE0gKyBpbnN0cnXDp8O1ZXMgKyBjb250ZXh0byArIHRvb2xzICsgZ3VhcmRyYWlscyArIGxvb3AiXSAtLT4gRFsiU2lzdGVtYSBhZ2VudGUiXQ?bgColor=FFFFFF" alt="Fluxograma 24" style="background:#fff;padding:8px;border-radius:6px">



Quando você olhar para `perguntar()` agora, tente identificar cada uma dessas peças no código.

## ✅ Resumo da aula

| Etapa | Função | Para que serve |
|---|---|---|
| Schema | `montar_schema()` | Conta ao LLM como é o banco |
| Gerar | `gerar_sql()` | LLM escreve o SQL |
| Validar | `validar_sql()` | 1ª defesa: só `SELECT`, 1 consulta, com `LIMIT` |
| Executar | `executar_sql()` | 2ª defesa: SQLite somente leitura + timeout |
| Corrigir | laço em `perguntar()` | LLM vê o próprio erro e tenta de novo |
| Explicar | `explicar()` | Tabela vira resposta em português |

**Desafios:** (1) adicione uma pergunta nova em `EXEMPLOS`; (2) tente quebrar o validador com um SQL malicioso; (3) peça algo que os dados não respondem e veja o `NAO_SEI` em ação.

